# Example 5.10

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

def Flow_2D_CD(N, nu, Uw, tau):
    """
    Solves 2D incompressible flow in a square cavity using the
    vorticity–streamfunction formulation (central differences).
    """

    # Grid parameters
    dx = 1.0 / N              # dx = dy
    Nx = N + 1
    dx2 = dx**2
    nudx2 = nu / dx2

    X = np.linspace(0, 1, Nx)
    Y = np.linspace(0, 1, Nx)

    # SOR parameters for Poisson equation
    sigma = np.cos(np.pi / Nx)
    omega = 2.0 / (1.0 + np.sqrt(1.0 - sigma**2))

    # Tridiagonal matrix for Poisson solver
    e0 = np.ones(Nx)
    e1 = np.ones(Nx - 1)

    A = diags(
    diagonals=[-e1, 4*e0, -e1],
    offsets=[-1, 0, 1],
    shape=(Nx, Nx),
    format="csc"
).toarray()

# Boundary rows
    A[0, :] = 0
    A[0, 0] = 1
    A[-1, :] = 0
    A[-1, -1] = 1

    # Initialize fields
    U = np.zeros((Nx, Nx))
    V = np.zeros((Nx, Nx))
    Psi = np.zeros((Nx, Nx))
    Z = np.zeros((Nx, Nx))

    # Top wall boundary condition
    U[:, -1] = Uw
    Z[:, -1] = -2 * Uw / dx

    # ODE system for vorticity
    def system(t, y):
        Zloc = y.reshape((Nx, Nx)).T
        Zp = np.zeros_like(Zloc)

        for i in range(1, Nx-1):
            for j in range(1, Nx-1):
                diffusion = nudx2 * (
                    Zloc[i-1, j] + Zloc[i+1, j]
                    + Zloc[i, j-1] + Zloc[i, j+1]
                    - 4 * Zloc[i, j]
                )
                convection = (
                    U[i, j] * (Zloc[i, j] - Zloc[i-1, j]) / dx +
                    V[i, j] * (Zloc[i, j] - Zloc[i, j-1]) / dx
                )
                Zp[i, j] = diffusion - convection

        return Zp.T.reshape(Nx * Nx)

    # Poisson solver for streamfunction
    def poisson(Z):
        P = Psi.copy()
        err = 1.0
        it = 0

        while err > 1e-3 and it < 100:
            Pold = P.copy()

            for j in range(1, Nx-1):
                b = np.zeros(Nx)
                for i in range(1, Nx-1):
                    b[i] = dx2 * Z[i, j] + P[i, j-1] + Pold[i, j+1]

                v = np.linalg.solve(A, b)
                P[:, j] = Pold[:, j] + omega * (v - Pold[:, j])

            err = np.linalg.norm(P - Pold) / np.linalg.norm(P)
            it += 1

        return P

    # Time loop
    for k in range(1, len(tau)):
        print(f"Time step {k}")

        Z0 = Z.T.reshape(Nx * Nx)

        sol = solve_ivp(
            system,
            (tau[k-1], tau[k]),
            Z0,
            method="LSODA",
            rtol=1e-3,
            atol=1e-4
        )

        Z = sol.y[:, -1].reshape((Nx, Nx)).T

        Psi[:] = poisson(Z)

        # Update velocities
        for i in range(1, Nx-1):
            for j in range(1, Nx-1):
                U[i, j] = 0.5 * (Psi[i, j+1] - Psi[i, j-1]) / dx
                V[i, j] = -0.5 * (Psi[i+1, j] - Psi[i-1, j]) / dx

        # Boundary conditions for vorticity
        for i in range(Nx):
            Z[i, 0] = 2 * (Psi[i, 0] - Psi[i, 1]) / dx2
            Z[i, -1] = 2 * (Psi[i, -1] - Psi[i, -2] - Uw * dx) / dx2

        for j in range(1, Nx-1):
            Z[0, j] = 2 * (Psi[0, j] - Psi[1, j]) / dx2
            Z[-1, j] = 2 * (Psi[-1, j] - Psi[-2, j]) / dx2

    # Visualization
    plt.figure(figsize=(10, 8))

    plt.subplot(2, 2, 1)
    plt.contourf(X, Y, Z.T, cmap="jet")
    plt.colorbar()
    plt.title(r"Vorticity $\zeta$")

    plt.subplot(2, 2, 2)
    plt.contourf(X, Y, Psi.T, cmap="jet")
    plt.colorbar()
    plt.title("Streamfunction $\psi$")

    plt.subplot(2, 2, 3)
    plt.contourf(X, Y, U.T, cmap="jet")
    plt.colorbar()
    plt.title("Velocity U")

    plt.subplot(2, 2, 4)
    plt.contourf(X, Y, V.T, cmap="jet")
    plt.colorbar()
    plt.title("Velocity V")

    plt.tight_layout()
    filename = 'Figure_5_13.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

    return U, V, Psi, Z

In [ ]:
# parameter setting 
N=20
nu=0.1
Uw=1
tau=np.linspace(0, 1, 100)
Flow_2D_CD(N, nu, Uw, tau);